In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

Using Device: cuda


In [5]:
transform = transforms.Compose([
    transforms.Resize((224,224)),   # same size images
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

In [6]:
dataset = datasets.ImageFolder(
    root="data",
    transform=transform
)

print("Classes:", dataset.classes)
print("Total Images:", len(dataset))

Classes: ['cloudy', 'desert', 'green_area', 'water']
Total Images: 5631


In [7]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [9]:
class CNN_Model(nn.Module):
    def __init__(self, num_classes=4):
        super(CNN_Model, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [10]:
model = CNN_Model().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
epochs = 10

for epoch in range(epochs):

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss:.4f} "
          f"Accuracy: {acc:.2f}%")

Epoch [1/10] Loss: 65.4735 Accuracy: 78.11%
Epoch [2/10] Loss: 43.2862 Accuracy: 87.74%
Epoch [3/10] Loss: 42.0677 Accuracy: 88.50%
Epoch [4/10] Loss: 37.8485 Accuracy: 89.72%
Epoch [5/10] Loss: 32.2453 Accuracy: 91.30%
Epoch [6/10] Loss: 31.3785 Accuracy: 91.74%
Epoch [7/10] Loss: 31.9127 Accuracy: 91.36%
Epoch [8/10] Loss: 29.2945 Accuracy: 91.98%
Epoch [9/10] Loss: 29.6169 Accuracy: 92.03%
Epoch [10/10] Loss: 29.7436 Accuracy: 92.41%


In [12]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:

        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", 100*correct/total)

Validation Accuracy: 93.2564330079858


In [14]:
from PIL import Image

img = Image.open(r"D:\Subjects_Languages\Languages\AICT_SureTrust_Internship\ALL_Assignments\MLP\Satellite\data\green_area\Forest_2.jpg")

img = transform(img).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    output = model(img)
    _, pred = torch.max(output,1)

print("Predicted Class:",
      dataset.classes[pred.item()])

Predicted Class: green_area
